In [1]:
import threading
import time
import urllib.request

import altair as alt
from tqdm import tqdm

In [2]:
REQUESTS_PER_CLIENT = 10_000
SCENARIOS = [1, 2, 5, 10]

In [3]:
def reset_counter(base_url):
    req = urllib.request.Request(f"{base_url}/reset", method="POST")
    urllib.request.urlopen(req)

In [4]:
def client_worker(base_url, n_requests, index, progress_bar):
    for _ in range(n_requests):
        urllib.request.urlopen(f"{base_url}/inc")
        progress_bar.update(1)

In [ ]:
def run_all(base_url, storage_label):

    print(f"\n=== {storage_label.upper()} ===")

    results = []

    for n in SCENARIOS:
        progress_bar = tqdm(
            total=n * REQUESTS_PER_CLIENT,
            desc=f"{storage_label} | {n} client(s)",
            unit="req",
            dynamic_ncols=True,
        )
        with progress_bar:
            threads = [
                threading.Thread(
                    target=client_worker,
                    args=(base_url, REQUESTS_PER_CLIENT, i, progress_bar),
                )
                for i in range(n)
            ]
            wall_start = time.time()
            for t in threads:
                t.start()
            for t in threads:
                t.join()
            wall_time = time.time() - wall_start
        results.append(
            {
                "clients": str(n),
                "storage": storage_label,
                "RPS": round(total_requests / wall_time, 1),
                "wall_time": round(wall_time, 3),
            }
        )
    return results

In [6]:
results = run_all("http://localhost:8080", "memory") + run_all(
    "http://localhost:8081", "disk"
)


=== MEMORY ===


memory | 1 client(s):   0%|          | 0/10000 [00:00<?, ?req/s]

memory | 1 client(s): 100%|██████████| 10000/10000 [00:06<00:00, 1522.51req/s]
memory | 2 client(s): 100%|██████████| 20000/20000 [00:07<00:00, 2539.02req/s]
memory | 5 client(s): 100%|██████████| 50000/50000 [00:17<00:00, 2844.37req/s]
memory | 10 client(s): 100%|██████████| 100000/100000 [00:34<00:00, 2894.20req/s]



=== DISK ===


disk | 1 client(s): 100%|██████████| 10000/10000 [01:10<00:00, 140.85req/s]
disk | 2 client(s): 100%|██████████| 20000/20000 [00:44<00:00, 449.60req/s]
disk | 5 client(s): 100%|██████████| 50000/50000 [01:53<00:00, 440.82req/s]
disk | 10 client(s): 100%|██████████| 100000/100000 [03:38<00:00, 457.13req/s]


In [13]:
def make_chart(
    y_field,
    y_title,
    chart_title,
):
    bars = (
        alt.Chart(alt.Data(values=results))
        .mark_bar()
        .encode(
            x=alt.X(
                "clients:O",
                title="clients",
                sort=["1", "2", "5", "10"],
                axis=alt.Axis(labelAngle=0),
            ),
            xOffset="storage:N",
            y=alt.Y(f"{y_field}:Q", title=y_title, aggregate="sum"),
            color=alt.Color(
                "storage:N",
                scale=alt.Scale(
                    domain=["memory", "disk"], range=["#3b82f6", "#f97316"]
                ),
                legend=alt.Legend(title=None, orient="bottom"),
            ),
        )
    )
    labels = bars.mark_text(dy=-8, fontSize=12).encode(
        text=alt.Text(f"{y_field}:Q", format=".1f", aggregate="sum"),
    )
    return (bars + labels).properties(title=chart_title, width=600)


# make charts
(
    make_chart("RPS", "RPS", "Throughput (RPS)")
    & make_chart("wall_time", "Wall Time (s)", "Wall Time (s)")
).show()


alt.VConcatChart(...)